In [ ]:
import open3d as o3d
import os
from util import (
    auto_select_sam3_prompt,
    flip_uv_v_inplace,
    save_sam_masks_multi_frames,
    select_viewpoints,
    render_mesh_materials,
    load_masks_for_mesh_frame,
    fuse_views_to_face_mask,
    build_face_adjacency,
    build_local_band_around_dynamic,
    local_expand_dynamic,
    split_mesh_by_face_mask,
    remove_small_dynamic_components
)

In [ ]:
dataset = 'synthetic'
out_dir = f"./data/{dataset}"
os.makedirs(out_dir, exist_ok=True)
mesh = o3d.io.read_triangle_mesh(os.path.join(out_dir, "meshes/frame_0000.obj"), enable_post_processing=True)
mesh.compute_vertex_normals()
#o3d.visualization.draw_geometries([mesh])
view_files_exist = all(os.path.exists(f"{out_dir}/view_{i:02d}.json") for i in range(4))
num_views = 20
if not (view_files_exist):
    viewpoints = select_viewpoints(mesh, mesh, num_views=num_views, width=1920, height=1080)
    for i, cam in enumerate(viewpoints):
        o3d.io.write_pinhole_camera_parameters(f"{out_dir}/view_{i:02d}.json", cam)


In [ ]:
import cv2
viewpoints = [o3d.io.read_pinhole_camera_parameters(f"{out_dir}/view_{i:02d}.json") for i in range(num_views)]

In [ ]:
import cv2
viewpoints = [o3d.io.read_pinhole_camera_parameters(f"{out_dir}/view_{i:02d}.json") for i in range(num_views)]
for i in range(0, 10):
    mesh = o3d.io.read_triangle_mesh(os.path.join(out_dir, f"meshes/frame_{i:04d}.obj"), enable_post_processing=True)
    flip_uv_v_inplace(mesh)
    render_dir = os.path.join(out_dir, "render")
    os.makedirs(render_dir, exist_ok=True)
    print(render_dir)

    for j, view in enumerate(viewpoints):
        mesh_texture, mesh_depth, mesh_depth_range = render_mesh_materials(mesh, width=1920, height=1080, camera_params=view, enable_lighting=False)
        cv2.imwrite(os.path.join(render_dir, f"{i:02d}_v_{j:02d}.jpg"), cv2.cvtColor(mesh_texture, cv2.COLOR_RGB2BGR))

In [ ]:
import os
import sam3
import torch

sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")

# use all available GPUs on the machine
gpus_to_use = range(torch.cuda.device_count())
# # use only a single GPU
# gpus_to_use = [torch.cuda.current_device()]

In [ ]:
from sam3.model_builder import build_sam3_video_predictor

predictor = build_sam3_video_predictor(gpus_to_use=gpus_to_use)

In [ ]:
import glob
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sam3.visualization_utils import (
    load_frame,
    prepare_masks_for_visualization,
    visualize_formatted_frame_output,
)

# font size for axes titles
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["figure.titlesize"] = 12


def propagate_in_video(predictor, session_id):
    # we will just propagate from frame 0 to the end of the video
    outputs_per_frame = {}
    for response in predictor.handle_stream_request(
        request=dict(
            type="propagate_in_video",
            session_id=session_id,
        )
    ):
        outputs_per_frame[response["frame_index"]] = response["outputs"]

    return outputs_per_frame


def abs_to_rel_coords(coords, IMG_WIDTH, IMG_HEIGHT, coord_type="point"):
    """Convert absolute coordinates to relative coordinates (0-1 range)

    Args:
        coords: List of coordinates
        coord_type: 'point' for [x, y] or 'box' for [x, y, w, h]
    """
    if coord_type == "point":
        return [[x / IMG_WIDTH, y / IMG_HEIGHT] for x, y in coords]
    elif coord_type == "box":
        return [
            [x / IMG_WIDTH, y / IMG_HEIGHT, w / IMG_WIDTH, h / IMG_HEIGHT]
            for x, y, w, h in coords
        ]
    else:
        raise ValueError(f"Unknown coord_type: {coord_type}")

In [ ]:
# "video_path" needs to be either a JPEG folder or a MP4 video file
video_path = f"./data/{dataset}/render"

In [ ]:
# load "video_frames_for_vis" for visualization purposes (they are not used by the model)
if isinstance(video_path, str) and video_path.endswith(".mp4"):
    cap = cv2.VideoCapture(video_path)
    video_frames_for_vis = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        video_frames_for_vis.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
else:
    video_frames_for_vis = glob.glob(os.path.join(video_path, "*.jpg"))
    try:
        # integer sort instead of string sort (so that e.g. "2.jpg" is before "11.jpg")
        video_frames_for_vis.sort(
            key=lambda p: int(os.path.splitext(os.path.basename(p))[0])
        )
    except ValueError:
        # fallback to lexicographic sort if the format is not "<frame_index>.jpg"
        print(
            f'frame names are not in "<frame_index>.jpg" format: {video_frames_for_vis[:5]=}, '
            f"falling back to lexicographic sort."
        )
        video_frames_for_vis.sort()

In [ ]:
response = predictor.handle_request(
    request=dict(
        type="start_session",
        resource_path=video_path,
    )
)
session_id = response["session_id"]

In [ ]:
# note: in case you already ran one text prompt and now want to switch to another text prompt
# it's required to reset the session first (otherwise the results would be wrong)
_ = predictor.handle_request(
    request=dict(
        type="reset_session",
        session_id=session_id,
    )
)

In [ ]:
prompt_text_str = ""
frame_idx = 0  # add a text prompt on frame 0
response = predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=session_id,
        frame_index=frame_idx,
        text=prompt_text_str,
    )
)
out = response["outputs"]

plt.close("all")
visualize_formatted_frame_output(
    frame_idx,
    video_frames_for_vis,
    outputs_list=[prepare_masks_for_visualization({frame_idx: out})],
    titles=["SAM 3 Dense Tracking outputs"],
    figsize=(6, 4),
)

In [ ]:
# now we propagate the outputs from frame 0 to the end of the video and collect all outputs
outputs_per_frame = propagate_in_video(predictor, session_id)

# finally, we reformat the outputs for visualization and plot the outputs every 60 frames
outputs_per_frame = prepare_masks_for_visualization(outputs_per_frame)

vis_frame_stride = 80
plt.close("all")
for frame_idx in range(0, len(outputs_per_frame), vis_frame_stride):
    visualize_formatted_frame_output(
        frame_idx,
        video_frames_for_vis,
        outputs_list=[outputs_per_frame],
        titles=["SAM 3 Dense Tracking outputs"],
        figsize=(6, 4),
    )

In [ ]:
auto_prompt = auto_select_sam3_prompt(
    video_path,
    prompt_frame_pos=0,
    num_positive_points=4,
    num_negative_points=0,
)

frame_idx = auto_prompt["flat_frame_index"]
obj_id = 0
points_abs = auto_prompt["points_abs"]
labels = auto_prompt["labels"]

sample_img = Image.fromarray(load_frame(video_frames_for_vis[frame_idx]))
IMG_WIDTH, IMG_HEIGHT = sample_img.size

print(
    f"Selected reference view {auto_prompt['reference_view_id']:02d} "
    f"on mesh frame {auto_prompt['frame_id']:02d} "
    f"(motion peak at t={auto_prompt['analysis_frame_pos']:02d}, flat frame index={frame_idx})"
)
print("View motion scores:", auto_prompt["view_scores"])


In [ ]:
auto_prompt_summary = {
    "reference_view_id": auto_prompt["reference_view_id"],
    "analysis_frame_pos": auto_prompt["analysis_frame_pos"],
    "prompt_frame_pos": auto_prompt["prompt_frame_pos"],
    "flat_frame_index": auto_prompt["flat_frame_index"],
    "num_prompt_points": int(len(points_abs)),
}
auto_prompt_summary


In [ ]:
ref_img = load_frame(video_frames_for_vis[frame_idx])

plt.close("all")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].imshow(ref_img)
pos_mask = labels == 1
if np.any(pos_mask):
    axes[0].scatter(points_abs[pos_mask, 0], points_abs[pos_mask, 1], c="lime", s=60, label="positive")
if np.any(labels == 0):
    axes[0].scatter(points_abs[labels == 0, 0], points_abs[labels == 0, 1], c="red", s=60, label="negative")
axes[0].set_title(f"Auto prompt on frame {frame_idx}")
axes[0].axis("off")
if len(points_abs) > 0:
    axes[0].legend(loc="lower right")

axes[1].imshow(auto_prompt["heatmap"], cmap="magma")
axes[1].imshow(auto_prompt["prompt_mask"], cmap="spring", alpha=0.35)
axes[1].set_title(f"Reference view {auto_prompt['reference_view_id']:02d} motion proposal")
axes[1].axis("off")

plt.tight_layout()


In [ ]:
# convert automatically selected points and labels to tensors
points_tensor = torch.tensor(
    abs_to_rel_coords(points_abs, IMG_WIDTH, IMG_HEIGHT, coord_type="point"),
    dtype=torch.float32,
)
points_labels_tensor = torch.tensor(labels, dtype=torch.int32)

response = predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=session_id,
        frame_index=frame_idx,
        points=points_tensor,
        point_labels=points_labels_tensor,
        obj_id=obj_id,
    )
)
out = response["outputs"]

plt.close("all")
visualize_formatted_frame_output(
    frame_idx,
    video_frames_for_vis,
    outputs_list=[prepare_masks_for_visualization({frame_idx: out})],
    titles=[f"SAM 3 auto-prompt output (view {auto_prompt['reference_view_id']:02d})"],
    figsize=(6, 4),
    points_list=[points_abs],
    points_labels_list=[labels],
)


In [ ]:
# propagate the auto-initialized mask through the flattened frame-view sequence
outputs_per_frame = propagate_in_video(predictor, session_id)

# reformat the outputs for visualization and plot roughly one sample per mesh frame
outputs_per_frame = prepare_masks_for_visualization(outputs_per_frame)

vis_frame_stride = 15
plt.close("all")
for frame_idx in range(0, len(outputs_per_frame), vis_frame_stride):
    visualize_formatted_frame_output(
        frame_idx,
        video_frames_for_vis,
        outputs_list=[outputs_per_frame],
        titles=["SAM 3 auto-initialized tracking outputs"],
        figsize=(6, 4),
    )


In [ ]:
print(
    f"Prepared {len(outputs_per_frame)} tracked masks across "
    f"{auto_prompt['render_index']['num_frames']} mesh frames and "
    f"{auto_prompt['render_index']['num_views']} views."
)


In [ ]:
mask_root = f"./data/{dataset}/masks"
save_sam_masks_multi_frames(outputs_per_frame, mask_root, num_frames=10, num_views=num_views)

In [ ]:
dyn_root = f"./data/{dataset}/meshes/dynamic"
stat_root = f"./data/{dataset}/meshes/static"
gt_root = f"./data/{dataset}/meshes/gt"

os.makedirs(dyn_root, exist_ok=True)
os.makedirs(stat_root, exist_ok=True)
os.makedirs(gt_root, exist_ok=True)

num_frames = 10
viewpoints = [o3d.io.read_pinhole_camera_parameters(f"{out_dir}/view_{i:02d}.json") for i in range(num_views)]
for t in range(num_frames):
    print(f"frame {t}")

    sam_masks_list = load_masks_for_mesh_frame(mask_root, t, num_views=num_views)

    mesh = o3d.io.read_triangle_mesh(os.path.join(out_dir, f"meshes/frame_{t:04d}.obj"))
    mesh.compute_vertex_normals()

    # save GT
    o3d.io.write_triangle_mesh(os.path.join(gt_root, f"frame_{t:04d}.obj"), mesh)

    face_mask_t, ratio_t, total_vis_t, unseen_t, low_vis_t = fuse_views_to_face_mask(
        mesh=mesh,
        camera_params_list=viewpoints,
        sam_masks_list=sam_masks_list,
        width=1920,
        height=1080,
        tau=0.6
    )

    face_mask_remove = remove_small_dynamic_components(
        face_is_dynamic=face_mask_t,
        neighbors=neighbors,
        min_component_faces=20
    )

    neighbors = build_face_adjacency(mesh)

    # Local region only: faces within 1 or 2 mesh-adjacency steps
    # from the initial dynamic region.
    candidate_band = build_local_band_around_dynamic(
        face_is_dynamic=face_mask_remove,
        neighbors=neighbors,
        band_iters=10
    )

    # Only consider faces that SAM could not reliably classify.
    expandable = (
        (~face_mask_remove)
        & candidate_band
        & (
            (total_vis_t <= 20) |
            (ratio_t >= 0.02)
        )
    )

    face_mask_refined = local_expand_dynamic(
        face_is_dynamic=face_mask_remove,
        neighbors=neighbors,
        candidate_band=candidate_band,
        expandable_mask=expandable,
        min_dynamic_neighbors=1,
        num_iters=20
    )

    dyn, stat = split_mesh_by_face_mask(mesh, face_mask_refined)

    # save dynamic
    o3d.io.write_triangle_mesh(os.path.join(dyn_root, f"frame_{t:04d}.obj"), dyn)

    # save static
    o3d.io.write_triangle_mesh(os.path.join(stat_root, f"frame_{t:04d}.obj"),stat,write_triangle_uvs=True)